# 09 — Analyse causale (évaluation de faisabilité)

## Décision d'applicabilité (B)
**NON APPLICABLE** (pour une inférence causale robuste sur ces données)

Cette étape évalue explicitement la distinction :
1. corrélation
2. association
3. prédiction
4. causalité

Nous calculons les trois premiers niveaux (corrélation, association, prédiction), puis nous documentons
pourquoi la causalité n'est pas identifiable de façon crédible ici.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from src.data import load_processed_dataset
from src.analysis import did_feasibility_report, save_table
from src.modeling import fit_ols, prepare_model_frame

df = load_processed_dataset("education_prioritaire")
print(df.shape)


(5250, 21)


## 1. Corrélation (niveau descriptif le plus faible)

In [2]:
corr_vars = ["variable_cible", "ips", "taille_moyenne_classe", "dedoublement", "rep", "rep_plus"]
corr_matrix = df[corr_vars].corr(numeric_only=True)
save_table(corr_matrix, "09_correlation_matrix")
corr_matrix


,variable_cible,ips,taille_moyenne_classe,dedoublement,rep,rep_plus
variable_cible,1.000000,-0.023754,-0.083360,-0.207845,-0.181139,-0.284260
ips,-0.023754,1.000000,0.012117,-0.011850,-0.035314,-0.001148
taille_moyenne_classe,-0.083360,0.012117,1.000000,-0.503114,-0.164298,-0.184397
dedoublement,-0.207845,-0.011850,-0.503114,1.000000,0.527303,0.438194
rep,-0.181139,-0.035314,-0.164298,0.527303,1.000000,-0.272797
rep_plus,-0.284260,-0.001148,-0.184397,0.438194,-0.272797,1.000000


**Interprétation (corrélation).** La corrélation mesure une co-variation brute entre deux variables.
Elle n'implique ni mécanisme, ni contrôle des facteurs confondants, ni causalité.

## 2. Association (régression observationnelle contrôlée)

In [3]:
explanatory = ["statut", "niveau", "ips", "dedoublement"]
categorical = ["statut", "niveau"]
frame = prepare_model_frame(df, "variable_cible", explanatory, categorical=categorical, group_column="ecole_id")
result_ols, formule = fit_ols(frame, "variable_cible", explanatory, categorical=categorical, cluster_column="ecole_id")
association_table = pd.DataFrame(
    {
        "coefficient": result_ols.params,
        "p_value": result_ols.pvalues,
    }
)
save_table(association_table, "09_association_ols_coefficients")
association_table


,coefficient,p_value
Intercept,94.706027,0.000000e+00
C(statut)[T.REP],-11.064153,1.994803e-195
C(statut)[T.REP+],-15.184389,0.000000e+00
C(niveau)[T.CE1],-14.970457,0.000000e+00
C(niveau)[T.CM1],-11.016286,0.000000e+00
C(niveau)[T.CM2],-6.078143,8.708374e-181
C(niveau)[T.CP],-22.191819,0.000000e+00
ips,-0.029340,1.964231e-04
dedoublement,6.108484,3.003145e-69


**Interprétation (association).** La régression contrôle certaines covariables (`ips`, `niveau`),
et mesure des associations conditionnelles. Cela améliore la comparaison par rapport à la corrélation,
mais reste non causal sans stratégie d'identification.

## 3. Prédiction (performance hors échantillon)

In [4]:
X = df[["statut", "niveau", "ips", "dedoublement", "taille_moyenne_classe", "effectif_eleves"]]
y = df["variable_cible"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

categorical_features = ["statut", "niveau"]
numeric_features = ["ips", "dedoublement", "taille_moyenne_classe", "effectif_eleves"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

pred_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LinearRegression()),
    ]
)

pred_model.fit(X_train, y_train)
y_pred = pred_model.predict(X_test)

prediction_metrics = pd.DataFrame(
    [
        {"metrique": "R2_test", "valeur": r2_score(y_test, y_pred)},
        {"metrique": "RMSE_test", "valeur": np.sqrt(mean_squared_error(y_test, y_pred))},
        {"metrique": "MAE_test", "valeur": mean_absolute_error(y_test, y_pred)},
    ]
)
save_table(prediction_metrics, "09_prediction_metrics")
prediction_metrics


,metrique,valeur
0,R2_test,0.715504
1,RMSE_test,5.586926
2,MAE_test,4.458009


**Interprétation (prédiction).** La prédiction évalue la capacité à anticiper `variable_cible`.
Même une bonne performance prédictive ne prouve jamais une relation causale : on peut bien prédire sans
expliquer le mécanisme de cause à effet.

## 4. Causalité : test de faisabilité d'une stratégie DiD

In [5]:
feasibility = did_feasibility_report(df)
feasibility_df = pd.DataFrame([feasibility])
save_table(feasibility_df, "09_faisabilite_causale_did")
feasibility_df


,applicable,share_schools_with_status_changes,mean_treatment_cp_ce1,mean_treatment_other_levels,policy_alignment_gap,has_controls_each_year,reasons_if_not_applicable
0,False,1.0,0.335238,0.335238,0.0,True,[Les statuts des écoles varient fortement dans...


In [6]:
reasons = feasibility["reasons_if_not_applicable"]
if reasons:
    print("Raisons du NON APPLICABLE causal :")
    for idx, reason in enumerate(reasons, start=1):
        print(f"{idx}. {reason}")
else:
    print("Tous les critères minimaux de faisabilité sont remplis.")


Raisons du NON APPLICABLE causal :
1. Les statuts des écoles varient fortement dans le temps (changement annuel fréquent), ce qui invalide l'interprétation d'un groupe traité stable.
2. L'exposition au dédoublement n'est pas spécifique à CP/CE1 dans ces données (écart CP/CE1 vs autres niveaux trop faible).


## 5. Conclusion causale

**CAUSALITÉ : NON APPLICABLE (sur ce dataset)**

### Pourquoi
- Les statuts des écoles changent trop souvent d'une année à l'autre, empêchant de définir des groupes
  traités/contrôles stables.
- L'exposition au dédoublement n'est pas spécifique à CP/CE1 dans les données simulées, ce qui contredit
  le protocole institutionnel attendu pour une DiD crédible.
- La donnée est simulée/reconstruite : elle sert à démontrer une démarche, pas à établir une preuve causale.

### Hypothèses causales non satisfaites
- stabilité des unités et du traitement ;
- trajectoires parallèles plausibles entre groupes comparables ;
- cohérence temporelle du déploiement.

### Portée de ce notebook
- Oui : distinction rigoureuse corrélation/association/prédiction/causalité ;
- Non : preuve causale d'un effet du dédoublement.
